# Extract data + Spectrogram generation



In [11]:
import audb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

# Load the emodb database
db = audb.load("emodb")

Get:   emodb v2.0.0
Cache: /root/audb/emodb/2.0.0/d3b62a9b


In [12]:
# get the data, no ambiguous audios, separate train and test splits

train_data: pd.DataFrame = db["emotion.categories.train.gold_standard"].get().copy()
test_data: pd.DataFrame = db["emotion.categories.test.gold_standard"].get().copy()

From the audios given, we make them 3s long by padding or truncating them. Then, we calculate their spectrogram and add it to the dataframe to make it usable.

In [13]:
def truncate_or_pad(x, target_len):
    current_len = x.shape[1]

    if isinstance(x, torch.Tensor):
        if current_len > target_len:
            return x[:, :target_len]
        else:
            pad_size = target_len - current_len
            pad_shape = (1, pad_size)
            padding = torch.zeros(pad_shape, dtype=x.dtype, device=x.device)
            return torch.cat([x, padding], dim=1)


def compute_spectrogram(file_path):
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = truncate_or_pad(waveform, sample_rate * 3)

    # 25ms windows every 10ms
    window_length = int(sample_rate * 0.025)  # 25ms frames
    hop_length = int(sample_rate * 0.010)  # 10ms hop

    mel_transform = T.MelSpectrogram(
        sample_rate=sample_rate,
        win_length=window_length,
        hop_length=hop_length,
        n_mels=64,
    )

    amp_to_db = T.AmplitudeToDB(stype="power", top_db=80)

    spec_db = amp_to_db(mel_transform(waveform)).squeeze(0)

    return spec_db


def add_spectrogram_col(datafram_data):
    spectrogram_list = []
    for file_path in datafram_data.index:
        spectrogram_list.append(compute_spectrogram(file_path))

    datafram_data.insert(1, "spectrogram", spectrogram_list)

add_spectrogram_col(train_data)
add_spectrogram_col(test_data)

Finally, we can verify the number of audios for each of the splits, and show some spectrograms we just computed.

In [14]:
print("Number of training audios: ", len(train_data))
print("Number of testing audios:  ", len(test_data))

# plt.imshow(
#     compute_spectrogram(train_data.index[1]).numpy(),
#     origin="lower",
#     aspect=301 / 64,
#     cmap="gray",
# )
# plt.axis("off")
# plt.show()
# plt.imshow(
#     compute_spectrogram(test_data.index[1]).numpy(),
#     origin="lower",
#     aspect=301 / 64,
#     cmap="gray",
# )
# plt.axis("off")
# plt.show()

Number of training audios:  304
Number of testing audios:   231


# "From scratch" NN
## 1. NN architecture definition
Let's first specify the architecture that the NN will use. This is done by defining a class that inherits nn.Module.

In [15]:
class fromScratch(nn.Module):
    def __init__(self, in_channels=1):  # in_channels = 1 because we deal with gray scale images
        super().__init__()

        # we could have been making a helper function so that we don't have to define all four blocks by hand but since it's our first time using torch I think it helps understanding the NN's architecture
        self.block1 = nn.Sequential(  # first block : going from 1 channel to 32
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(  # second block : going from 32 channels to 64
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=64,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block3 = nn.Sequential(  # third block : going from 64 channels to 128
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=128,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block4 = nn.Sequential(  # fourth block : going from 128 channels to 256
            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=256,
                out_channels=256,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))  # global average pooling : this performs an average on each of the 256 small images produced, so that we go from 256 small matrices to 256 digits
        self.dropout = nn.Dropout(p=0.5)  # during training, half of the 256 values produced (chosen randomly) will be set to zero, which helps avoid overfitting
        self.fc = nn.Linear(in_features=256, out_features=7)  # going from 256 values to 7, since our goal is to classify each audio in 7 emotions


    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

## 2. Dataset and dataloader definition
In order to feed data to our newly defined NN, we rely on torch's Dataset and DataLoader structures.

In [16]:
classes = sorted(train_data['emotion'].unique())  # extracting all 7 emotions from the "train_data" DataFrame
label2id = {emotion: i for i, emotion in enumerate(classes)}  # each emotion gets a label from 0 to 6

# Dataset definition
class EmotionDataset(Dataset):
    def __init__(self, df, label_mapping):
        self.spectrograms = df['spectrogram'].values
        self.labels = [label_mapping[emo] for emo in df['emotion']]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        spec = self.spectrograms[idx].unsqueeze(0).float()  # up to this point spectrograms are of shape [64,301], pytorch requires that we add one dimension so that it becomes [1,64,301]
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return spec, label

# Dataloaders definition
train_dataset = EmotionDataset(train_data, label2id)
test_dataset = EmotionDataset(test_data, label2id)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # shuffling is mandatory to prevent the NN from learning the order of the data
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 3. Training preparation
Using GPU when available, and choosing a loss function and an optimizer.

In [17]:
# If GPU is available, it will be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on : {device}")

model = fromScratch(in_channels=1).to(device)

criterion = nn.CrossEntropyLoss()  # this is the function that will be used to compute the loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # the optimizer updates the weights following the gradient computation

Training on : cuda


## 4. Training functions definition
Utility functions for the upcoming training step. The first function trains the model over one epoch (*ie* over the whole train set), the second one evaluates it in its current state over the whole test set.

In [18]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # the model is put in training mode (dropout activated)
    running_loss, correct, total = 0.0, 0, 0

    for specs, labels in loader:  # each loop turn retrieves one batch
        specs, labels = specs.to(device), labels.to(device)

        # steps to train the model on current batch
        optimizer.zero_grad()  # reset gradients from previous batch
        outputs = model(specs)  # forward pass : computes the model's answer to the current batch following current weights
        loss = criterion(outputs, labels)  # computes the difference between predicted classes and actual labels
        loss.backward()  # gradient computation
        optimizer.step()  # weights update

        # tracking loss
        running_loss += loss.item() * specs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()  # the model is put in evaluation mode, which disables dropout
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for specs, labels in loader:
            specs, labels = specs.to(device), labels.to(device)
            outputs = model(specs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * specs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

## 5. Model training
Training the model over a chosen number of epochs

In [20]:
EPOCHS = 50
best_test_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), "fromScratch_trained.pth")

    print(f"Epoch [{epoch:02d}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.1f}% | "
          f"Test Loss: {test_loss:.4f}, Acc: {test_acc*100:.1f}%")

print(f"\nBest precision on test set : {best_test_acc*100:.2f}%.")

Epoch [01/50] | Train Loss: 0.0527, Acc: 97.7% | Test Loss: 3.6424, Acc: 51.1%
Epoch [02/50] | Train Loss: 0.0720, Acc: 97.7% | Test Loss: 2.0198, Acc: 53.2%
Epoch [03/50] | Train Loss: 0.0524, Acc: 99.0% | Test Loss: 1.5514, Acc: 61.0%
Epoch [04/50] | Train Loss: 0.0300, Acc: 100.0% | Test Loss: 1.5447, Acc: 66.7%
Epoch [05/50] | Train Loss: 0.0399, Acc: 99.3% | Test Loss: 4.4220, Acc: 38.1%
Epoch [06/50] | Train Loss: 0.0204, Acc: 100.0% | Test Loss: 13.8711, Acc: 16.9%
Epoch [07/50] | Train Loss: 0.0386, Acc: 99.7% | Test Loss: 1.1553, Acc: 69.3%
Epoch [08/50] | Train Loss: 0.0288, Acc: 99.7% | Test Loss: 1.5664, Acc: 58.0%
Epoch [09/50] | Train Loss: 0.0808, Acc: 97.7% | Test Loss: 10.0411, Acc: 23.8%
Epoch [10/50] | Train Loss: 0.0509, Acc: 99.3% | Test Loss: 5.3119, Acc: 49.8%
Epoch [11/50] | Train Loss: 0.0525, Acc: 99.0% | Test Loss: 4.1847, Acc: 42.0%
Epoch [12/50] | Train Loss: 0.0537, Acc: 98.4% | Test Loss: 2.7885, Acc: 54.1%
Epoch [13/50] | Train Loss: 0.0661, Acc: 98.7% |